## 🛠️ 1. Setup & Utilities

In [ ]:
# 1.1 Environment Setup

!apt-get update && apt-get install -y libsndfile1 ffmpeg
!pip install -q wget text-unidecode
!pip install -q "numpy==2.0.2" "scipy==1.13.1" speechbrain nemo_toolkit[asr] torchaudio librosa

In [ ]:
# 1.2 Imports

import os
import gc
import time
import random
import glob
import math
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

import torchaudio
import nemo.collections.asr as nemo_asr
from speechbrain.inference.speaker import EncoderClassifier
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN
from speechbrain.lobes.features import Fbank
import soundfile as sf

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

In [ ]:
# 1.3 Find previous executions

LOAD_DIR = "/kaggle/input/datasets/davidozzo/test-tentativo"
SAVE_DIR = "./"

def sync_from_dataset(relative_path):
    """Needed for crash management"""
    load_path = os.path.join(LOAD_DIR, relative_path)
    save_path = os.path.join(SAVE_DIR, relative_path)
    if os.path.exists(load_path) and not os.path.exists(save_path):
        print(f"Syncing {relative_path} from dataset to working dir...")
        if os.path.isdir(load_path):
            shutil.copytree(load_path, save_path)
            for root, dirs, files in os.walk(save_path):
                for d in dirs:
                    os.chmod(os.path.join(root, d), 0o777)
                for f in files:
                    os.chmod(os.path.join(root, f), 0o666)
        else:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            shutil.copy2(load_path, save_path)
            os.chmod(save_path, 0o666)

sync_from_dataset("eval_split.csv")
sync_from_dataset("librispeech_data")
sync_from_dataset("voxceleb_train.csv")
sync_from_dataset("voxceleb_eval.csv")
sync_from_dataset("precomputed_embeddings")
sync_from_dataset("pretrained_models")
sync_from_dataset("checkpoint_latest.pt")
sync_from_dataset("best_model.pt")
sync_from_dataset("checkpoint_latest_v2.pt")
sync_from_dataset("best_model_v2.pt")
sync_from_dataset("checkpoint_latest_v3.pt")
sync_from_dataset("best_model_v3.pt")

BASE_WORK_DIR = SAVE_DIR

In [ ]:
# 1.4 Utility Functions

def seed_everything(seed=42):
    """Set seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def compute_eer_mindcf(y_true, y_scores, p_target=0.01, c_miss=1, c_fa=1):
    """Metrics for speaker verification"""
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    
    eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
    eer = fpr[eer_idx]
    eer_threshold = thresholds[eer_idx] 
    
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    mindcf_idx = np.argmin(dcf)
    mindcf = dcf[mindcf_idx]
    optimal_threshold = thresholds[mindcf_idx]
    
    default_dcf = min(c_miss * p_target, c_fa * (1 - p_target))
    mindcf_normalized = mindcf / default_dcf
    
    return eer, mindcf_normalized, eer_threshold

def generate_trial_pairs(df, num_pos_pairs=50000, num_neg_pairs=50000):
    """Make pairs of samples with the same speaker for evaluation"""
    speakers = df.groupby('speaker_id')['file_path'].apply(list).to_dict()
    speaker_keys = list(speakers.keys())
    
    trials = []
    
    pos_count = 0
    while pos_count < num_pos_pairs:
        spk = random.choice(speaker_keys)
        paths = speakers[spk]
        if len(paths) > 1:
            p1, p2 = random.sample(paths, 2)
            trials.append((p1, p2, 1))
            pos_count += 1
            
    neg_count = 0
    while neg_count < num_neg_pairs:
        spk1, spk2 = random.sample(speaker_keys, 2)
        p1 = random.choice(speakers[spk1])
        p2 = random.choice(speakers[spk2])
        trials.append((p1, p2, 0))
        neg_count += 1
        
    return trials

def count_params(model):
    """Useful for printing parameters"""
    if hasattr(model, "parameters"):
        try:
            return sum(p.numel() for p in model.parameters())
        except TypeError:
            pass
    if hasattr(model, "mods"):
        return sum(p.numel() for p in model.mods.parameters())
    raise ValueError(f"Don't know how to count parameters for {type(model)}")

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}\n")

def precompute_collate_fn(batch):
    """Useful for precomputing embeddings"""
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None, None, None
    waveforms = [item[0] for item in batch]
    paths = [item[1] for item in batch]
    lengths = torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long)
    padded = pad_sequence(waveforms, batch_first=True)
    return padded, lengths, paths

def eval_collate_fn(batch):
    """Collate for EER during evaluation"""
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return [], []
    waves = [item[0] for item in batch]
    paths = [item[1] for item in batch]
    return waves, paths

## 🏛️ 2. Architectures

This cell defines the student model and the classification layer.

*   **`MultiScaleAttentiveStatsPooling` Class**: Replaces the encoder's single-scale ASP; pools attentive mean/std at several temporal granularities (global + local) and fuses them.

*   **`StudentECAPA` Class**: Defines the student model, a scaled-down version of the original ECAPA-TDNN, with a multi-scale pooling head and a feature-level distillation head for the ECAPA teacher.

*   **`AAMSoftmax` Class**: Implements the Additive Angular Margin Softmax layer.

In [ ]:
# 2.1 Architectures

class MultiScaleAttentiveStatsPooling(nn.Module):
    """
    Implements a Multi-Scale Attentive Statistics Pooling layer
    with a bottleneck to prevent parameter explosion during fusion
    """
    def __init__(self, channels, attention_channels=128, scales=(1, 2, 4), global_context=True):
        super().__init__()
        self.scales = scales
        self.global_context = global_context
        
        # Calculate input dimension dynamically
        in_dim = channels * 3 if global_context else channels
        self.tdnn = nn.Conv1d(in_dim, attention_channels, kernel_size=1)
        self.tanh = nn.Tanh()
        self.conv = nn.Conv1d(attention_channels, channels, kernel_size=1)
        
        # Bottleneck to compress the concatenated dimensions before fusion
        concat_dim = channels * 2 * len(scales)
        bottleneck_dim = channels
        
        self.fuse = nn.Sequential(
            nn.Conv1d(concat_dim, bottleneck_dim, kernel_size=1, groups=len(scales)),
            nn.BatchNorm1d(bottleneck_dim),
            nn.GELU(),
            nn.Conv1d(bottleneck_dim, channels * 2, kernel_size=1)
        )

    def _masked_stats(self, x, mask):
        valid_counts = mask.sum(dim=2, keepdim=True).clamp(min=1)
        mean = (x * mask).sum(dim=2, keepdim=True) / valid_counts

        if self.global_context:
            std = (((x - mean) ** 2) * mask).sum(dim=2, keepdim=True) / valid_counts
            std = std.clamp(min=1e-4).sqrt()
            attn_input = torch.cat([x, mean.expand_as(x), std.expand_as(x)], dim=1)
        else:
            attn_input = x

        attn = self.tanh(self.tdnn(attn_input))
        attn = self.conv(attn)
        attn = attn.masked_fill(mask == 0, -1e4)
        attn = F.softmax(attn, dim=2)

        w_mean = torch.sum(x * attn, dim=2)
        w_var = (torch.sum((x ** 2) * attn, dim=2) - w_mean ** 2).clamp(min=1e-4)
        w_std = torch.sqrt(w_var)
        return torch.cat([w_mean, w_std], dim=1).unsqueeze(2)

    def forward(self, x, lengths=None):
        B, C, T = x.shape
        if lengths is None:
            mask = torch.ones(B, 1, T, device=x.device, dtype=x.dtype)
        else:
            valid_len = (lengths.float() * T).long().clamp(min=1, max=T)
            idx = torch.arange(T, device=x.device).view(1, 1, T)
            mask = (idx < valid_len.view(B, 1, 1)).to(x.dtype)

        pooled = []
        for n_segments in self.scales:
            seg_len = max(1, T // n_segments)
            seg_stats = []
            for i in range(n_segments):
                start = i * seg_len
                end = T if i == n_segments - 1 else start + seg_len
                seg_stats.append(self._masked_stats(x[:, :, start:end], mask[:, :, start:end]))
            pooled.append(torch.stack(seg_stats, dim=0).mean(dim=0))

        fused = torch.cat(pooled, dim=1)
        return self.fuse(fused)


class StudentECAPA(nn.Module):
    def __init__(self, scale_fraction=1, input_dim=80, embedding_dim=192):
        super().__init__()
        
        self.compute_features = Fbank(n_mels=input_dim)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=50)
        
        self.scale_fraction = scale_fraction
        base_channels = [512, 512, 512, 512, 1536]
        self.scaled_channels = [max(16, (int(c * scale_fraction) // 8) * 8) for c in base_channels]
        
        self.encoder = ECAPA_TDNN(
            input_size=input_dim,
            channels=self.scaled_channels,
            lin_neurons=embedding_dim,
        )

        self.encoder.asp = MultiScaleAttentiveStatsPooling(
            channels=self.scaled_channels[-1],
            attention_channels=128,
            scales=(1, 2, 4),
            global_context=True,
        )

        def _capture_mfa(module, inp, output):
            module._captured_features = output
            
        self.encoder.mfa.register_forward_hook(_capture_mfa)
        
        def build_projection_head(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, in_dim * 2),
                nn.BatchNorm1d(in_dim * 2),
                nn.GELU(),
                nn.Linear(in_dim * 2, out_dim)
            )
        
        self.head_ecapa = build_projection_head(embedding_dim, 192)
        self.head_titanet = build_projection_head(embedding_dim, 192)
        self.head_ecapa_feat = None

    def build_feat_head(self, teacher_feat_dim, device):
        mfa_channels = self.scaled_channels[-1]
        self.head_ecapa_feat = nn.Sequential(
            nn.Conv1d(mfa_channels, mfa_channels, kernel_size=1),
            nn.BatchNorm1d(mfa_channels),
            nn.GELU(),
            nn.Conv1d(mfa_channels, teacher_feat_dim, kernel_size=1)
        ).to(device)
        
    def forward(self, x, lengths=None):
        x = self.compute_features(x)
        x = x - x.mean(dim=1, keepdim=True)

        if self.training:
            x = x.transpose(1, 2)
            x = self.freq_mask(x)
            x = self.time_mask(x)
            x = x.transpose(1, 2)
        
        student_emb = self.encoder(x, lengths)
        student_emb = student_emb.squeeze(1) 
        
        proj_ecapa = self.head_ecapa(student_emb)
        proj_titanet = self.head_titanet(student_emb)

        proj_ecapa_feat = None
        mfa_features = getattr(self.encoder.mfa, '_captured_features', None)
        if self.head_ecapa_feat is not None and mfa_features is not None:
            proj_ecapa_feat = self.head_ecapa_feat(mfa_features).mean(dim=2)
        
        return {
            "student_core": student_emb,
            "proj_ecapa": proj_ecapa,   
            "proj_titanet": proj_titanet,
            "proj_ecapa_feat": proj_ecapa_feat
        }

    def get_param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class AAMSoftmax(nn.Module):
    """Loss function for highly discriminative embeddings"""
    def __init__(self, in_features, out_features, scale=64.0, margin=0.20):
        super(AAMSoftmax, self).__init__()
        self.scale = scale
        self.margin = margin
        
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    @torch.amp.autocast(device_type='cuda', enabled=False)
    def forward(self, embeddings, labels):
        embeddings = embeddings.float()
        weight = self.weight.float()
        
        cosine = F.linear(F.normalize(embeddings), F.normalize(weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2).clamp(1e-7, 1.0 - 1e-7))
        
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros(cosine.size(), device=embeddings.device)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.scale

## 🗂️ 3. Data Preparation: MUSAN, LibriSpeech & VoxCeleb

The first cell sets up the auxiliary datasets used for noise augmentation and evaluation.

*   **MUSAN Dataset**.
*   **LibriSpeech Dataset**.

The second cell processes the **VoxCeleb Dataset** used for training the student model and for evaluation.

In [ ]:
# 3.1 Data Preparation: MUSAN & LibriSpeech
 
print("Preparing MUSAN dataset for noise augmentation...")
MUSAN_PATH = "/kaggle/input/datasets/nhattruongdev/musan-noise" 
eval_noise_files = glob.glob(f"{MUSAN_PATH}/**/*.wav", recursive=True)
 
print(f"MUSAN: {len(eval_noise_files)} noise files available for evaluation.")
 
LIBRI_EVAL_CSV = os.path.join(BASE_WORK_DIR, "eval_split.csv")
LIBRISPEECH_ROOT = os.path.join(BASE_WORK_DIR, "librispeech_data")
LIBRISPEECH_URL = "train-clean-100"
 
print("Preparing LibriSpeech CSV for cross-dataset testing...")
if os.path.exists(LIBRI_EVAL_CSV):
    eval_df_libri = pd.read_csv(LIBRI_EVAL_CSV)
    print(f"LibriSpeech Eval loaded from cache: {len(eval_df_libri)} samples.")
else:
    try:
        os.makedirs(LIBRISPEECH_ROOT, exist_ok=True)
        print(f"Downloading LibriSpeech {LIBRISPEECH_URL}...")
        librispeech_dataset = torchaudio.datasets.LIBRISPEECH(
            root=LIBRISPEECH_ROOT,
            url=LIBRISPEECH_URL,
            download=True
        )
        print(f"Successfully loaded {len(librispeech_dataset)} LibriSpeech samples.")
 
        extracted_dir = os.path.join(LIBRISPEECH_ROOT, "LibriSpeech", LIBRISPEECH_URL)
        all_flacs = glob.glob(f"{extracted_dir}/**/*.flac", recursive=True)
 
        data = [
            {"file_path": path, "speaker_id": os.path.basename(path).split('-')[0]}
            for path in all_flacs
        ]
        eval_df_libri = pd.DataFrame(data)
        eval_df_libri.to_csv(LIBRI_EVAL_CSV, index=False)
        print(f"LibriSpeech Eval built and cached: {len(eval_df_libri)} samples.")
 
    except Exception as e:
        print(f"WARNING: could not download/prepare LibriSpeech ({e}).")
        eval_df_libri = pd.DataFrame()

In [ ]:
# 3.2 Data Preparation: VoxCeleb
 
VOXCELEB_PATH = "/kaggle/input/datasets/dhruvkarmokar/voxceleb-dataset" 
TRAIN_CSV = os.path.join(BASE_WORK_DIR, "voxceleb_train.csv")
EVAL_CSV = os.path.join(BASE_WORK_DIR, "voxceleb_eval.csv")
 
if os.path.exists(TRAIN_CSV) and os.path.exists(EVAL_CSV):
    print("Loading existing VoxCeleb splits...")
    train_df = pd.read_csv(TRAIN_CSV)
    eval_df = pd.read_csv(EVAL_CSV)
else:
    print("Scanning VoxCeleb directory...")
    all_wavs = glob.glob(f"{VOXCELEB_PATH}/**/*.wav", recursive=True)
    
    data = []
    for path in all_wavs:
        parts = path.split('/')
        speaker_id = next((p for p in parts if p.startswith('id')), None)
        if speaker_id:
            data.append({"file_path": path, "speaker_id": speaker_id})
            
    df = pd.DataFrame(data)
    
    unique_speakers = sorted(df['speaker_id'].unique())
    train_speakers, eval_speakers = train_test_split(
        unique_speakers, 
        test_size=0.1, 
        random_state=42
    )
    
    train_df = df[df['speaker_id'].isin(train_speakers)].copy()
    eval_df = df[df['speaker_id'].isin(eval_speakers)].copy()
    
    speaker_to_idx = {spk: idx for idx, spk in enumerate(sorted(train_speakers))}
    train_df['encoded_label'] = train_df['speaker_id'].map(speaker_to_idx)
    eval_df['encoded_label'] = -1
    
    train_df.to_csv(TRAIN_CSV, index=False)
    eval_df.to_csv(EVAL_CSV, index=False)
 
print(f"VoxCeleb Train: {len(train_df)} utterances | Eval: {len(eval_df)} utterances")
n_classes = len(train_df['speaker_id'].unique())
n_eval_speakers = len(eval_df['speaker_id'].unique())
print(f"Unique Train Speakers: {n_classes}")
print(f"Unique Eval Speakers: {n_eval_speakers}")

## 💾 4. Pre-computation of Teacher Embeddings

This cell optimizes the distillation process by pre-computing the teacher embeddings, plus a
pooled intermediate feature map from the ECAPA teacher for feature-level distillation. Precomputations proved necessary because the computation was extremely long.

In [ ]:
# 4.1 Pre-computation of Teacher Embeddings
 
EMBEDDINGS_DIR = os.path.join(BASE_WORK_DIR, "precomputed_embeddings")
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
ECAPA_MMAP = os.path.join(EMBEDDINGS_DIR, "ecapa.npy")
TITANET_MMAP = os.path.join(EMBEDDINGS_DIR, "titanet.npy")
ECAPA_FEAT_MMAP = os.path.join(EMBEDDINGS_DIR, "ecapa_feat.npy")
INDEX_MAP = os.path.join(EMBEDDINGS_DIR, "path_index.json")
META_MAP = os.path.join(EMBEDDINGS_DIR, "mmap_meta.json")
 
class EmbeddingPrecomputeDataset(Dataset):
    """Dataset of precomputed embeddings"""
    def __init__(self, df, min_samples=16000 * 3, max_samples=16000 * 10):
        self.paths = df['file_path'].tolist()
        self.min_samples = min_samples
        self.max_samples = max_samples
 
    def __len__(self):
        return len(self.paths)
 
    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            data, sr = sf.read(path, dtype='float32', always_2d=True)
            waveform = torch.from_numpy(data).mean(dim=1)
            if waveform.shape[0] < self.min_samples:
                return None
            if waveform.shape[0] > self.max_samples:
                waveform = waveform[:self.max_samples]
            return waveform, path
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None

def precompute_embeddings_memmap(df, ecapa_model, titanet_model, device, batch_size=32, num_workers=4):
    """Computes embeddings + pooled ECAPA feature map and saves through memmap for optimization"""
    total_files = len(df)
    
    print(f"Allocating space for {total_files} embeddings on disk...")
    ecapa_fp = np.memmap(ECAPA_MMAP, dtype='float16', mode='w+', shape=(total_files, 192))
    titanet_fp = np.memmap(TITANET_MMAP, dtype='float16', mode='w+', shape=(total_files, 192))

    # Hook the ECAPA teacher's pre-pooling MFA output for feature-level distillation.
    captured_feat = {}

    def _feat_hook(module, inp, output):
        captured_feat["value"] = output

    hook_handle = ecapa_model.mods.embedding_model.mfa.register_forward_hook(_feat_hook)
    ecapa_feat_fp = None
    ecapa_feat_dim = None
    
    dataset = EmbeddingPrecomputeDataset(df)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=precompute_collate_fn,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    path_to_idx = {}
    current_idx = 0
    
    with torch.no_grad():
        for padded, lengths, paths in tqdm.tqdm(loader, total=len(loader)):
            if padded is None:
                continue
 
            try:
                padded = padded.to(device, non_blocking=True)
                lengths = lengths.to(device, non_blocking=True)
                lengths_f = lengths.float()
                rel_lengths = lengths_f / lengths_f.max()
 
                emb_e = ecapa_model.encode_batch(padded, wav_lens=rel_lengths).squeeze(1)
                _, emb_t = titanet_model(input_signal=padded, input_signal_length=lengths)

                # Mask out padded frames, then average-pool the captured MFA feature map over time.
                feat = captured_feat["value"]
                T_feat = feat.shape[2]
                valid_len = (rel_lengths * T_feat).long().clamp(min=1, max=T_feat)
                idx_range = torch.arange(T_feat, device=device).view(1, 1, T_feat)
                feat_mask = (idx_range < valid_len.view(-1, 1, 1)).to(feat.dtype)
                feat_pooled = (feat * feat_mask).sum(dim=2) / feat_mask.sum(dim=2).clamp(min=1)

                if ecapa_feat_fp is None:
                    ecapa_feat_dim = feat_pooled.shape[1]
                    ecapa_feat_fp = np.memmap(ECAPA_FEAT_MMAP, dtype='float16', mode='w+', shape=(total_files, ecapa_feat_dim))
 
                emb_e = emb_e.cpu().numpy().astype(np.float16)
                emb_t = emb_t.cpu().numpy().astype(np.float16)
                feat_pooled = feat_pooled.cpu().numpy().astype(np.float16)
 
            except Exception as e:
                print(f"Error processing batch: {e}")
                torch.cuda.empty_cache()
                continue
 
            for i, path in enumerate(paths):
                ecapa_fp[current_idx] = emb_e[i]
                titanet_fp[current_idx] = emb_t[i]
                ecapa_feat_fp[current_idx] = feat_pooled[i]
                path_to_idx[path] = current_idx
                current_idx += 1
 
            if current_idx % 10000 < batch_size:
                ecapa_fp.flush()
                titanet_fp.flush()
                ecapa_feat_fp.flush()

    hook_handle.remove()
    ecapa_fp.flush()
    titanet_fp.flush()
    if ecapa_feat_fp is not None:
        ecapa_feat_fp.flush()
    
    with open(INDEX_MAP, 'w') as f:
        json.dump(path_to_idx, f)
        
    with open(META_MAP, 'w') as f:
        json.dump({"total_rows": total_files, "valid_rows": current_idx, "ecapa_feat_dim": ecapa_feat_dim}, f)
        
    success_rate = (current_idx / total_files) if total_files > 0 else 0
    print(f"Pre-computed {current_idx} valid embeddings successfully to disk ({success_rate*100:.1f}% of {total_files} candidates).")
    if success_rate < 0.9:
        print(f"WARNING: only {success_rate*100:.1f}% of files were embedded.")
 
def check_embeddings_ready(min_success_rate=0.9):
    files_exist = (
        os.path.exists(ECAPA_MMAP) and os.path.exists(TITANET_MMAP) and os.path.exists(ECAPA_FEAT_MMAP)
        and os.path.exists(INDEX_MAP) and os.path.exists(META_MAP)
    )
    if not files_exist:
        return False
    with open(META_MAP, 'r') as f:
        meta = json.load(f)
    total_rows = meta.get("total_rows", 0)
    valid_rows = meta.get("valid_rows", total_rows)  # older metadata (pre-fix) doesn't have valid_rows
    if total_rows == 0 or meta.get("ecapa_feat_dim") is None:
        return False
    success_rate = valid_rows / total_rows
    if success_rate < min_success_rate:
        print(f"WARNING: cached embeddings incomplete ({valid_rows}/{total_rows} = {success_rate*100:.1f}%). Forcing re-computation.")
        return False
    return True

embeddings_ready = check_embeddings_ready()
 
if not embeddings_ready:
    print("Loading teacher models for pre-computation...")
    ecapa_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb", 
    run_opts={"device": str(device)},
    savedir=os.path.join(BASE_WORK_DIR, "pretrained_models", "ecapa")
    )
    titanet_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large").to(device)
    ecapa_model.eval()
    titanet_model.eval()
 
    precompute_embeddings_memmap(train_df, ecapa_model, titanet_model, device)
 
    del ecapa_model, titanet_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Memory-mapped embeddings already exist. Skipping pre-computation.")

## 🗃️ 5. Dataset and DataLoader

This cell defines the data pipeline used during the student's training phase.

In [ ]:
# 5.1 Dataset and DataLoader
 
class NoiseAugmenter(nn.Module):
    def __init__(self, noise_files, device, max_cached_noises=500, min_snr_db=5.0, max_snr_db=20.0, target_sr=16000):
        super().__init__()
        self.min_snr_db = min_snr_db
        self.max_snr_db = max_snr_db
        self._num_noises = 0
 
        sampled_files = random.sample(noise_files, min(max_cached_noises, len(noise_files)))
        for f in sampled_files:
            try:
                data, sr = sf.read(f, dtype='float32', always_2d=True)
                wave = torch.from_numpy(data).mean(dim=1)
                if sr != target_sr:
                    wave = torchaudio.functional.resample(wave, sr, target_sr)
                self.register_buffer(f"noise_{self._num_noises}", wave.to(device))
                self._num_noises += 1
            except Exception:
                continue
 
    @property
    def noises(self):
        return [getattr(self, f"noise_{i}") for i in range(self._num_noises)]
 
    @torch.no_grad()
    def forward(self, clean_batch):
        batch_size, target_len = clean_batch.shape
        device = clean_batch.device
 
        noise_list = []
        for _ in range(batch_size):
            selected_noise = random.choice(self.noises)
            noise_len = selected_noise.shape[0]
 
            if noise_len < target_len:
                repeats = (target_len // noise_len) + 1
                selected_noise = selected_noise.repeat(repeats)
                noise_len = selected_noise.shape[0]
 
            start_idx = random.randint(0, noise_len - target_len)
            noise_list.append(selected_noise[start_idx : start_idx + target_len])
 
        noise_batch = torch.stack(noise_list)
        clean_power = torch.mean(clean_batch ** 2, dim=1, keepdim=True)
        noise_power = torch.mean(noise_batch ** 2, dim=1, keepdim=True).clamp(min=1e-8)
 
        snr_db = torch.empty(batch_size, 1, device=device).uniform_(self.min_snr_db, self.max_snr_db)
        scale = torch.sqrt((clean_power / (10.0 ** (snr_db / 10.0))) / noise_power)
        scale = torch.where(clean_power == 0, torch.zeros_like(scale), scale)
 
        return clean_batch + (noise_batch * scale)
 
 
class VoxCeleb_Distillation(Dataset):
    """Uses memmap to look up and fetch the embeddings + pooled ECAPA feature for real-time distillation"""
    def __init__(self, df, embeddings_dir, chunk_duration_sec=3.0, sample_rate=16000):
        self.df = df
        self.chunk_samples = int(chunk_duration_sec * sample_rate)
        self.sample_rate = sample_rate
        
        index_path = os.path.join(embeddings_dir, "path_index.json")
        meta_path = os.path.join(embeddings_dir, "mmap_meta.json")
        
        with open(index_path, 'r') as f:
            self.path_to_idx = json.load(f)
            
        with open(meta_path, 'r') as f:
            meta = json.load(f)
            self.mmap_shape = (meta["total_rows"], 192)
            self.feat_dim = meta["ecapa_feat_dim"]
            self.feat_mmap_shape = (meta["total_rows"], self.feat_dim)
 
        self.ecapa_mmap_path = os.path.join(embeddings_dir, "ecapa.npy")
        self.titanet_mmap_path = os.path.join(embeddings_dir, "titanet.npy")
        self.ecapa_feat_mmap_path = os.path.join(embeddings_dir, "ecapa_feat.npy")
        
        self.ecapa_fp = None
        self.titanet_fp = None
        self.ecapa_feat_fp = None
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        if self.ecapa_fp is None:
            self.ecapa_fp = np.memmap(self.ecapa_mmap_path, dtype='float16', mode='r', shape=self.mmap_shape)
            self.titanet_fp = np.memmap(self.titanet_mmap_path, dtype='float16', mode='r', shape=self.mmap_shape)
            self.ecapa_feat_fp = np.memmap(self.ecapa_feat_mmap_path, dtype='float16', mode='r', shape=self.feat_mmap_shape)
    
        row = self.df.iloc[idx]
        path = row['file_path']
        label = row['encoded_label']
    
        info = sf.info(path)
        num_frames = info.frames
    
        if num_frames <= self.chunk_samples:
            data, _ = sf.read(path, dtype='float32', always_2d=True)
            waveform = torch.from_numpy(data).mean(dim=1)
            waveform = F.pad(waveform, (0, self.chunk_samples - num_frames))
        else:
            frame_offset = random.randint(0, num_frames - self.chunk_samples)
            data, _ = sf.read(path, start=frame_offset, frames=self.chunk_samples,
                               dtype='float32', always_2d=True)
            waveform = torch.from_numpy(data).mean(dim=1)
    
        emb_idx = self.path_to_idx[path]
        ecapa_target = torch.from_numpy(np.array(self.ecapa_fp[emb_idx])).float()
        titanet_target = torch.from_numpy(np.array(self.titanet_fp[emb_idx])).float()
        ecapa_feat_target = torch.from_numpy(np.array(self.ecapa_feat_fp[emb_idx])).float()

        ecapa_target = F.normalize(ecapa_target, p=2, dim=0)
        titanet_target = F.normalize(titanet_target, p=2, dim=0)
        ecapa_feat_target = F.normalize(ecapa_feat_target, p=2, dim=0)
    
        return waveform, ecapa_target, titanet_target, ecapa_feat_target, path, label
 
 
def fast_collate_fn(batch):
    clean_waves = torch.stack([item[0] for item in batch])
    ecapa_batch = torch.stack([item[1] for item in batch])
    titanet_batch = torch.stack([item[2] for item in batch])
    ecapa_feat_batch = torch.stack([item[3] for item in batch])
    paths = [item[4] for item in batch]
    labels_batch = torch.tensor([item[5] for item in batch], dtype=torch.long)
 
    return clean_waves, ecapa_batch, titanet_batch, ecapa_feat_batch, paths, labels_batch
 
 
print("Initializing dataset...")
 
train_dataset = VoxCeleb_Distillation(
    df=train_df,
    embeddings_dir=EMBEDDINGS_DIR,
    chunk_duration_sec=3.0,
    sample_rate=16000
)
 
valid_indices = [i for i, path in enumerate(train_dataset.df['file_path']) if path in train_dataset.path_to_idx]
train_dataset.df = train_dataset.df.iloc[valid_indices].reset_index(drop=True)
print(f"Filtered dataset down to {len(train_dataset.df)} valid training samples.")
 
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    collate_fn=fast_collate_fn,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
)

## 🎓 6. Knowledge Distillation Training

This cell contains the core training routine for the student model, utilizing knowledge distillation and standard speaker classification.

*   **GPU Support**: Introduces a `DPTrainingWrapper` that encapsulates both the `StudentECAPA` model and the `AAMSoftmax` classification layer.
*   **Distillation losses**: Embedding-level distillation (ECAPA + TitaNet) and feature-level distillation (ECAPA) use **cosine similarity** against the precomputed teacher targets, with dynamically adjusted weights based on epoch progression.
*   **Optimizer, Scheduler & AMP**: Employs the `AdamW` optimizer coupled with a **`CosineAnnealingLR`** scheduler to smoothly decay the learning rate over time. It also initializes a `GradScaler` to enable Automatic Mixed Precision.
*   **Resumable Training**: Implements a checkpointing system with warm-start support and an early stopping mechanism based on a composite validation metric.

In [ ]:
# 6.1 Training Loop

with open(META_MAP, 'r') as f:
    ecapa_feat_dim = json.load(f)["ecapa_feat_dim"]

student_model = StudentECAPA(scale_fraction=0.5)
student_model.build_feat_head(ecapa_feat_dim, device)
aam_layer = AAMSoftmax(in_features=192, out_features=n_classes, scale=30.0, margin=0.20)
criterion_ce = nn.CrossEntropyLoss()
criterion_distill = nn.SmoothL1Loss()

# Some configuration
OLD_BEST_CKPT_PATH = os.path.join(BASE_WORK_DIR, "best_model_v2.pt")
WARM_START_FROM = OLD_BEST_CKPT_PATH if os.path.exists(OLD_BEST_CKPT_PATH) else None
FRESH_START = False           # In case I want to reset the optimizer
TITANET_SHARE = 0.55          # I'm trying to give more weight to Tita
FEAT_SHARE = 0.5              # weight of the ECAPA feature-level term, relative to the ECAPA embedding-level term
IMPROVEMENT_THRESHOLD = 0.001 # Minimum improvement for patience
AAM_DOWN_SCALE = 0.1

if WARM_START_FROM:
    print(f"Attempting warm start from previous run {WARM_START_FROM} ...")
    try:
        student_model.load_state_dict(torch.load(WARM_START_FROM, map_location=device, weights_only=True))
    except RuntimeError as e:
        print(f"Warm start skipped, checkpoint is incompatible with the new architecture: {e}")

class DPTrainingWrapper(nn.Module):
    """Encapsulates the student model and aam layer to parallelize and balance compute and VRAM usage across nodes"""
    def __init__(self, student, aam):
        super().__init__()
        self.student = student
        self.aam = aam

    def forward(self, x, lengths, labels):
        outputs = self.student(x, lengths)
        logits = self.aam(outputs["student_core"], labels)
        return outputs["proj_ecapa"], outputs["proj_titanet"], outputs["proj_ecapa_feat"], logits

wrapper = DPTrainingWrapper(student_model, aam_layer).to(device)

if torch.cuda.device_count() > 1:
    print(f"Found {torch.cuda.device_count()} GPUs. Activating DataParallel.")
    wrapper = nn.DataParallel(wrapper)

optimizer = optim.AdamW(
    wrapper.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs = 150
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

start_epoch = 0
best_loss = float('inf')
patience = 10
patience_counter = 0

latest_ckpt_path = os.path.join(BASE_WORK_DIR, "checkpoint_latest_v3.pt")
best_ckpt_path = os.path.join(BASE_WORK_DIR, "best_model_v3.pt") # Related to sec. 7

if os.path.exists(latest_ckpt_path) and not FRESH_START:
    print("Checkpoint found. Resuming training...")
    checkpoint = torch.load(latest_ckpt_path, map_location=device)
    base_wrapper = wrapper.module if isinstance(wrapper, nn.DataParallel) else wrapper
    base_wrapper.load_state_dict(checkpoint['wrapper_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['best_loss']
    patience_counter = checkpoint['patience_counter']
    print(f"Resumed from epoch {start_epoch} | Best (monitor) Loss: {best_loss:.4f}")
elif os.path.exists(latest_ckpt_path) and FRESH_START:
    print("Ignoring the old optimizer because FRESH_START is True.")

wrapper.train()
print("\nStarting Training...")

for epoch in range(start_epoch, epochs):
    epoch_loss, epoch_loss_e, epoch_loss_t, epoch_loss_feat, epoch_loss_a = 0.0, 0.0, 0.0, 0.0, 0.0

    progress = epoch / max(epochs - 1, 1)
    cosine_progress = 0.5 * (1 - math.cos(math.pi * progress))

    D_START, D_END = 0.35, 0.20
    distill_weight = D_START - (D_START - D_END) * cosine_progress
    aam_weight = 1.0 - 2 * distill_weight

    w_ecapa = distill_weight * 2 * (1 - TITANET_SHARE)
    w_titanet = distill_weight * 2 * TITANET_SHARE
    w_ecapa_feat = w_ecapa * FEAT_SHARE

    data_wait_accum = 0.0
    batch_start = time.time()
    for batch_idx, (clean_waves, ecapa_targets, titanet_targets, ecapa_feat_targets, paths, labels) in enumerate(train_loader):
        data_wait_accum += time.time() - batch_start

        clean_waves = clean_waves.to(device, non_blocking=True)
        ecapa_targets = ecapa_targets.to(device, non_blocking=True)
        titanet_targets = titanet_targets.to(device, non_blocking=True)
        ecapa_feat_targets = ecapa_feat_targets.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        rel_lengths = torch.ones(clean_waves.size(0), device=device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            proj_ecapa, proj_titanet, proj_ecapa_feat, aam_logits = wrapper(clean_waves, rel_lengths, labels)


            loss_ecapa = (1 - F.cosine_similarity(proj_ecapa.float(), ecapa_targets, dim=1)).mean()
            loss_titanet = (1 - F.cosine_similarity(proj_titanet.float(), titanet_targets, dim=1)).mean()
            loss_ecapa_feat = (1 - F.cosine_similarity(proj_ecapa_feat.float(), ecapa_feat_targets, dim=1)).mean()
            loss_aam = criterion_ce(aam_logits, labels)

            total_loss = (w_ecapa * loss_ecapa) + (w_titanet * loss_titanet) + (w_ecapa_feat * loss_ecapa_feat) + (aam_weight * AAM_DOWN_SCALE * loss_aam)

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += total_loss.item()
        epoch_loss_e += loss_ecapa.item()
        epoch_loss_t += loss_titanet.item()
        epoch_loss_feat += loss_ecapa_feat.item()
        epoch_loss_a += loss_aam.item()

        if (batch_idx + 1) % 500 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | "
                  f"LR: {current_lr:.6f} | Total Loss: {total_loss.item():.4f} "
                  f"(E: {loss_ecapa.item():.4f}, T: {loss_titanet.item():.4f}, Feat: {loss_ecapa_feat.item():.4f}, AAM: {loss_aam.item():.4f}) | "
                  f"Data wait (500 batches): {data_wait_accum:.1f}s")
            data_wait_accum = 0.0

        batch_start = time.time()

    avg_loss = epoch_loss / len(train_loader)
    avg_loss_e = epoch_loss_e / len(train_loader)
    avg_loss_t = epoch_loss_t / len(train_loader)
    avg_loss_feat = epoch_loss_feat / len(train_loader)
    avg_loss_a = epoch_loss_a / len(train_loader)

    scheduler.step()

    monitor_metric = avg_loss_e + avg_loss_t + avg_loss_feat + (avg_loss_a * AAM_DOWN_SCALE)

    print("-" * 65)
    print(f"EPOCH {epoch+1} COMPLETED | Avg Total Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"Distill W (ECAPA/TitaNet/ECAPA-Feat): {w_ecapa:.3f}/{w_titanet:.3f}/{w_ecapa_feat:.3f} | AAM W: {aam_weight:.3f}")
    print(f"Distill ECAPA: {avg_loss_e:.4f} | Distill TitaNet: {avg_loss_t:.4f} | Distill ECAPA-Feat: {avg_loss_feat:.4f} | AAM: {avg_loss_a:.4f} | Monitor: {monitor_metric:.4f}")
    print("-" * 65)

    gc.collect()
    torch.cuda.empty_cache()

    base_wrapper = wrapper.module if isinstance(wrapper, nn.DataParallel) else wrapper

    torch.save({
        'epoch': epoch,
        'wrapper_state_dict': base_wrapper.state_dict(),
        'model_state_dict': base_wrapper.student.state_dict(),
        'aam_state_dict': base_wrapper.aam.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_loss': best_loss,
        'patience_counter': patience_counter
    }, latest_ckpt_path)

    if monitor_metric < best_loss * (1 - IMPROVEMENT_THRESHOLD):
        best_loss = monitor_metric
        patience_counter = 0
        torch.save(base_wrapper.student.state_dict(), best_ckpt_path)
        print("New best loss achieved. Model saved.")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

## 📊 7. Evaluation

This cell benchmarks the fully trained student model against its heavy teacher models using unseen data, including an in-domain test (VoxCeleb) and an out-of-domain test (LibriSpeech) to assess generalization.

*   **Performance Metrics**: It computes the cosine similarity for each trial pair and calculates standard speaker verification metrics: Equal Error Rate (EER) and Minimum Detection Cost Function (MinDCF).
*   **Final Reporting**: The script compares the parameter counts alongside the evaluation metrics for all models.

In [ ]:
# 7.1 Cross-Dataset Evaluation
import time

class EvalDataset(Dataset):
    """A dataset for evaluation that loads raw audio and converts it to mono"""
    def __init__(self, paths, min_samples=16000 * 3, max_samples=16000 * 6):
        self.paths = paths
        self.min_samples = min_samples
        self.max_samples = max_samples
 
    def __len__(self):
        return len(self.paths)
 
    def __getitem__(self, idx):
        path = self.paths[idx]
        data, _ = sf.read(path, dtype='float32', always_2d=True)
        wave = torch.from_numpy(data).mean(dim=1)

        if wave.shape[0] < self.min_samples:
            return None
        
        if wave.shape[0] > self.max_samples:
            wave = wave[:self.max_samples]
        return wave, path

# Function for extracting embeddings in batches
def extract_embeddings(models, unique_paths, noise_augmenter, device, batch_size=64):
    dataset = EvalDataset(unique_paths)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=eval_collate_fn, num_workers=4, pin_memory=False)
    
    for model in models.values():
        model.eval()

    emb_caches = {model_name: {} for model_name in models}
    inf_times = {model_name: 0.0 for model_name in models}
    total_samples = 0
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        for waves, paths in tqdm.tqdm(loader, desc="Extracting embeddings from all models"):
            
            try:
                waves_gpu = [w.to(device, non_blocking=True) for w in waves]
                
                # Apply noise to each wave before padding, only if a noise augmenter was provided
                if noise_augmenter is not None:
                    processed_waves_list = []
                    for w in waves_gpu:
                        w_2d = w.unsqueeze(0)
                        noisy_w = noise_augmenter(w_2d).squeeze(0)
                        processed_waves_list.append(noisy_w)
                else:
                    processed_waves_list = waves_gpu
                
                # Calculate lengths
                lengths = torch.tensor([w.size(0) for w in processed_waves_list], dtype=torch.float32, device=device)
                
                # Padding
                processed_waves = pad_sequence(processed_waves_list, batch_first=True)
                
                total_samples += len(paths)

                for model_type, model in models.items():
                    torch.cuda.synchronize()
                    start_time = time.time()

                    if model_type == "ecapa":
                        rel_lengths = lengths / lengths.max()
                        embs = model.encode_batch(processed_waves, wav_lens=rel_lengths).squeeze(1)
                    elif model_type == "titanet":
                        _, embs = model(input_signal=processed_waves, input_signal_length=lengths.long())
                    elif model_type == "student":
                        rel_lengths = lengths / lengths.max()
                        # Pass rel_lengths to the student model to mask the padding
                        outputs = model(processed_waves, rel_lengths)
                        embs = outputs["student_core"]
                    else:
                        continue
                    
                    torch.cuda.synchronize()
                    inf_times[model_type] += (time.time() - start_time)
                    
                    embs_norm = F.normalize(embs, p=2, dim=1).cpu()
                    for i, p in enumerate(paths):
                        emb_caches[model_type][p] = embs_norm[i]

            except Exception as e:
                print(f"Error processing eval batch ({len(paths)} files): {e}")
                torch.cuda.empty_cache()
                continue
                
    avg_inf_times_ms = {m: (t / total_samples) * 1000 if total_samples > 0 else 0 for m, t in inf_times.items()}
    return emb_caches, avg_inf_times_ms

# Function for computing metrics with all models
def evaluate_all_models(models, trials, noise_augmenter, device, batch_size=64):
    unique_paths = list(set([t[0] for t in trials] + [t[1] for t in trials]))
    
    emb_caches, inf_times = extract_embeddings(models, unique_paths, noise_augmenter, device, batch_size)
    
    results = {}
    for model_name, emb_cache in emb_caches.items():
        scores = []
        labels = []
        skipped = 0
        
        for p1, p2, label in trials:
            if p1 not in emb_cache or p2 not in emb_cache:
                skipped += 1
                continue
            emb1 = emb_cache[p1]
            emb2 = emb_cache[p2]
            
            sim = F.cosine_similarity(emb1, emb2, dim=0).item()
            scores.append(sim)
            labels.append(label)
        
        if skipped > 0:
            print(f"WARNING [{model_name}]: {skipped}/{len(trials)} trial pairs skipped.")
            
        eer, mindcf, threshold = compute_eer_mindcf(labels, scores)
        results[model_name] = (eer, mindcf, threshold, inf_times[model_name])
        
    return results

# Clean memory
try:
    del train_loader, train_dataset, optimizer, scheduler, scaler
    del student_model, aam_layer
    if 'wrapper' in locals():
        del wrapper
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
 
print("\nLoading best student model...")
with open(META_MAP, 'r') as f:
    ecapa_feat_dim = json.load(f)["ecapa_feat_dim"]

student_eval = StudentECAPA(scale_fraction=0.5).to(device)
student_eval.build_feat_head(ecapa_feat_dim, device)
if os.path.exists(best_ckpt_path):
    student_eval.load_state_dict(torch.load(best_ckpt_path, map_location=device, weights_only=True))
student_eval.eval()
 
print("Loading teacher models...")
ecapa_eval = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb", 
    run_opts={"device": str(device)},
    savedir=os.path.join(BASE_WORK_DIR, "pretrained_models", "ecapa")
)
titanet_eval = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained("nvidia/speakerverification_en_titanet_large")
titanet_eval = titanet_eval.to(device)
titanet_eval.eval()
 
eval_noise_augmenter = NoiseAugmenter(eval_noise_files, device=device, max_cached_noises=300)
 
datasets_to_test = {
    "VoxCeleb": eval_df,
}
if not eval_df_libri.empty:
    datasets_to_test["LibriSpeech"] = eval_df_libri
 
models_to_test = {
    "ecapa": ecapa_eval,
    "titanet": titanet_eval,
    "student": student_eval
}
 
model_param_counts = {name: count_params(model) for name, model in models_to_test.items()}
for name, count in model_param_counts.items():
    print(f"{name.upper()} parameters: {count:,} ({count / 1e6:.2f}M)")
 
final_results = []
 
for dataset_name, df_eval in datasets_to_test.items():
    print(f"\n--- Testing on {dataset_name} ---")
    
    trials = generate_trial_pairs(df_eval, num_pos_pairs=20000, num_neg_pairs=20000)
    
    # Dynamic noise is applied only for the LibriSpeech evaluation
    apply_noise = (dataset_name == "LibriSpeech")
    
    results = evaluate_all_models(
        models=models_to_test,
        trials=trials,
        noise_augmenter=eval_noise_augmenter if apply_noise else None,
        device=device,
        batch_size=32
    )
    
    for model_name, (eer, mindcf, threshold, inf_time) in results.items():
        final_results.append({
            "Dataset": dataset_name,
            "Model": model_name.upper(),
            "Params (M)": f"{model_param_counts[model_name] / 1e6:.2f}",
            "Inf. Time/Sample (ms)": f"{inf_time:.2f}",
            "EER (%)": f"{eer*100:.2f}%",
            "MinDCF": f"{mindcf:.4f}",
            "Opt Threshold (EER)": f"{threshold:.4f}" # Because I prefer EER for the application
        })
 
print("\n" + "="*60)
print("FINAL EVALUATION OUTCOMES")
print("="*60)
results_df = pd.DataFrame(final_results)
display(results_df)
 
results_df.to_csv(os.path.join(BASE_WORK_DIR, "final_evaluation_results.csv"), index=False)